# CS 3110/5990: Data Privacy

## Homework 4


In [1]:
# Load the data and libraries
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt


def laplace_mech(v, sensitivity, epsilon):
    return v + np.random.laplace(loc=0, scale=sensitivity / epsilon)


def pct_error(orig, priv):
    return np.abs(orig - priv) / orig * 100.0


adult = pd.read_csv(
    "https://github.com/jnear/cs3110-data-privacy/raw/main/homework/adult_with_pii.csv"
)

## Question 1 (10 points)

Complete the definition of `dp_sum_capgain` below. Your definition should compute a differentially private sum of the "Capital Gain" column of the `adult` dataset, and have a total privacy cost of `epsilon`.


In [2]:
def dp_sum_capgain(epsilon):
    # We define this upper bound semi-arbitrarily. If I were to look at the data
    # to decide this value I would fail to maintain differential privacy.
    # And if I were to conduct a query to help decide what this bound should be,
    # I would incure additional privacy cost. (which can be worth it)
    upper_bound = 250000

    # Compute the sum first, utilizing clipping so we can bound the sensitivity
    sum_capgain = adult["Capital Gain"].clip(lower=0, upper=upper_bound).sum()

    # Then we apply the laplace mechanism to the clipped sum to ensure differential privacy.
    sum_capgain_dp = laplace_mech(
        sum_capgain, sensitivity=upper_bound, epsilon=epsilon
    )
    return sum_capgain_dp


dp_sum_capgain(1.0)

np.float64(35132378.777868055)

In [3]:
# TEST CASE for question 1

real_sum = adult["Capital Gain"].sum()
r1 = np.mean([pct_error(real_sum, dp_sum_capgain(0.1)) for _ in range(100)])
r2 = np.mean([pct_error(real_sum, dp_sum_capgain(1.0)) for _ in range(100)])
r3 = np.mean([pct_error(real_sum, dp_sum_capgain(10.0)) for _ in range(100)])

print("Average errors:", r1, r2, r3)

assert r1 > 0
assert r2 > 0
assert r3 > 0
assert r1 < 10
assert r2 < 2
assert r3 < 0.2

Average errors: 7.798014671103482 0.6304423443865413 0.06415637577452339


## Question 2 (10 points)

In 2-5 sentences each, answer the following:

- What clipping parameter did you use in your definition of `dp_sum_capital`, and why?
- What was the sensitivity of the query you used in `dp_sum_capital`, and how is it bounded?
- Argue that your definition of `dp_sum_capital` has a total privacy cost of `epsilon`


- I chose an upper bound clipping parameter of $250,000. I chose this
  number as an arbitrary upper bound based solely on my prior knowledge of annual
  income distributions in America. I am only hoping this chosen bound will cover
  a majority of the values within the dataset, but cannot guarantee that. I decided not to query the
  actual data to inform this decision because I did not want to incur the
  additional privacy cost that an additional query would add.
- Without clipping, a summation query would have an unbounded sensitivity value.
  Because I have clipped the individual values to 250,000, **the sensitivity value
  here is 250,000**.
- The summation query happens before the laplace mechanism is applied. The
  laplace mechanism is applied a single time to the total summation value, using
  the supplied `epsilon` value once. Therefore the total privacy cost of my
  function here is exactly `epsilon`.


## Question 3 (10 points)

Complete the definition of `dp_avg_capgain` below. Your definition should compute a differentially private average (mean) of the "Capital Gain" column of the adult dataset, and have a **total privacy cost of epsilon**.


In [4]:
def dp_avg_capgain(epsilon):
    upper_bound = 250000

    # Compute the sum first, utilizing clipping so we can bound the sensitivity
    sum_capgain = adult["Capital Gain"].clip(lower=0, upper=upper_bound).sum()

    # Then we apply the laplace mechanism to the clipped sum to ensure
    # differential privacy. Using epsilon/2 for the sum and epsilon/2 for the
    # count, to maintain that the total privacy cost stays at epsilon.
    sum_capgain_dp = laplace_mech(
        sum_capgain, sensitivity=upper_bound, epsilon=epsilon / 2
    )

    # We compute the count, as a differentially private version with Laplace noise.
    count = len(adult)
    count_dp = laplace_mech(count, sensitivity=1, epsilon=epsilon / 2)

    return sum_capgain_dp / count_dp


dp_avg_capgain(1.0)

np.float64(1068.7447127358766)

In [5]:
# TEST CASE for question 3

real_avg = adult["Capital Gain"].mean()
r1 = np.mean([pct_error(real_avg, dp_avg_capgain(0.1)) for _ in range(100)])
r2 = np.mean([pct_error(real_avg, dp_avg_capgain(1.0)) for _ in range(100)])
r3 = np.mean([pct_error(real_avg, dp_avg_capgain(10.0)) for _ in range(100)])

print("Average errors:", r1, r2, r3)

assert r1 > 0
assert r2 > 0
assert r3 > 0
assert r1 < 20
assert r2 < 4
assert r3 < 0.4

Average errors: 13.358471047417686 1.584252940262972 0.14417005922281184


## Question 4 (10 points)

In 2-5 sentences each, answer the following:

- Argue that your definition of `dp_avg_capgain` has a total privacy cost of `epsilon`
- For sums and averages, which seems to be more important for accuracy - the value of the clipping parameter $b$ or the scale of the noise added? Why?
- Do you think the answer to the previous point will be true for every dataset? Why or why not?


- The total privacy cost of `dp_avg_capgain` is `epsilon` because the function
  splits the privacy budget evenly across the two laplace mechanism usages. The sum is computed using
  the Laplace mechanism with `epsilon/2`, and the count is also computed using
  `epsilon/2`. The total privacy cost is computed as the sum of the two by
  "sequential composition". Therefore total cost is `epsilon/2 + epsilon/2 = epsilon`.

- For sums and averages, the clipping parameter `b` is usually more important
  for accuracy than the amount of noise, especially if the dataset has large
  outliers. The noise is controlled by the sensitivity and privacy budget, but
  `b` directly determines the sensitivity. If `b` is too small, the query is
  biased; if it is too large, the sensitivity grows and the noise becomes much
  worse.

- The answer is not always true for every dataset because it depends on how
  spread out the data is. If the values are tightly clustered, the noise may
  matter more than the clipping bound. But if the data is heavy-tailed or
  contains extreme values, then the clipping parameter would be the bigger factor.


## Question 5 (20 points)

Write a function `auto_avg` that returns the differentially private average of a Pandas series `s`. Your function should automatically determine the clipping parameter `b`, and should enforce differential privacy for a **total privacy cost** of `epsilon`. You can assume that all values are non-negative (i.e. 0 or greater).


In [6]:
def auto_avg(s, epsilon):
    # Increase candidate clipping values by 25% until reaching a fixed bound.
    absolute_upper_bound = 1000000
    bs = [1.0]
    while bs[-1] < absolute_upper_bound:
        bs.append(min(bs[-1] * 1.25, absolute_upper_bound))

    # Spend half of epsilon on choosing the clipping bound.
    epsilon_i = (epsilon / 2) / len(bs)
    last_sum = 0

    for b in bs:
        clipped_sum = s.clip(lower=0, upper=b).sum()
        dp_sum_test = laplace_mech(
            clipped_sum, sensitivity=b, epsilon=epsilon_i
        )

        if dp_sum_test < last_sum:
            chosen_b = b
            break
        last_sum = dp_sum_test
    else:
        chosen_b = bs[-1]

    # Spend the remaining half evenly between the sum and count queries.
    dp_sum = laplace_mech(
        s.clip(lower=0, upper=chosen_b).sum(),
        sensitivity=chosen_b,
        epsilon=epsilon / 4,
    )
    count_dp = laplace_mech(len(s), sensitivity=1, epsilon=epsilon / 4)

    return dp_sum / count_dp

In [7]:
# TEST CASE for question 5
r1 = np.mean(
    [
        pct_error(adult["Age"].mean(), auto_avg(adult["Age"], 1.0))
        for _ in range(20)
    ]
)
r2 = np.mean(
    [
        pct_error(
            adult["Capital Gain"].mean(), auto_avg(adult["Capital Gain"], 1.0)
        )
        for _ in range(20)
    ]
)
r3 = np.mean(
    [
        pct_error(adult["fnlwgt"].mean(), auto_avg(adult["fnlwgt"], 1.0))
        for _ in range(20)
    ]
)

print("Average errors:", r1, r2, r3)
assert r1 > 0
assert r2 > 0
assert r3 > 0
assert r1 < 1
assert r2 < 100
assert r3 < 1

Average errors: 0.048099723354941296 65.10532246624702 0.5594785911460656


## Question 6

In 2-5 sentences each, answer the following:

- Explain your strategy for implementing `auto_avg`
- Argue informally that your definition of `auto_avg` has a total privacy cost of `epsilon`
- Did your solution work well for all three example columns? If it did not work well on any of them, why not?
- When is your solution likely to _not_ work well? (i.e. what properties does the data have to have, in order for your solution to not work well?)


- My strategy for `auto_avg` was to start with a clipping value of 1, multiply
  it by 1.25 each time, and choose the first value where the noisy clipped sum
  starts to decrease (going until I hit an absolute upper bound of 1000000, to
  avoid checking forever). Then I use that bound for the final private average.

- The total privacy cost is `epsilon`. Half of the budget is split across the
  candidate-bound queries, and the remaining half is split evenly between the
  final sum and count queries. Sequential composition gives `epsilon/2 +
epsilon/4 + epsilon/4 = epsilon`.

- This worked well for Age and `fnlwgt`, with average errors below 1%. It did
  not work as well for Capital Gain, where the average error was about 65%, even
  though it technically passed the test's 100% threshold.

- My solution tests more values on the lower end of the spectrum between 1 and
  'absolute_upper_bound = 1000000', due to the nature of a geometric sequence
  vs. an arithmetic sequence. Therefore datasets where a good portion of the
  values present are relatively larger, then my specific strategy here may chose
  a bounding value that is a bit too far from the ideal. The issue here is that
  we need to directly look at the data (or at least what kind of data we are
  observing) to make a smarter guess. Additionally, my absolute upper bound was
  also chosen arbitrarily, and could also be a cause for failure.
